# Product Quantization tự viết và chọn index theo ngân sách

Bài mở: bạn tự cài Product Quantization (PQ) với tìm kiếm ADC, rồi dựng một bảng đo đủ rộng (exact, IVF, HNSW, PQ,
IVF-PQ, có và không rescoring) trên 61 425 vector `bge-m3` của Zalo Legal, vẽ biên Pareto recall / độ trễ / bộ nhớ, và
chọn cấu hình cho ba kịch bản trong `starter/REPORT.md`.

- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`. Phần đo là của bạn: thêm cell `explore` tuỳ ý.
- Cần cache `data/raw/zalo-legal/cache/bge-m3.npy` (bài Intermediate 01) và Ollama `bge-m3` để embed 300 câu hỏi.
- Chấm hàm: `uv run pytest tracks/ai-engineer/modules/12-embeddings-vector-db/exercises/advanced/01-ann-benchmark-pq -q`

In [ ]:
# explore: tìm gốc repo, đường dẫn dữ liệu
import os
import sys
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "4")
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure

ZALO = ensure("zalo-legal")
CACHE = ZALO / "cache"

In [ ]:
import numpy as np

## 1. Product Quantization

Chia vector $$d$$ chiều thành $$m$$ đoạn liên tiếp, mỗi đoạn $$d/m$$ chiều. Mỗi đoạn có một codebook $$k_s$$ centroid học
bằng k-means trên đoạn đó của dữ liệu train. Một vector được lưu bằng $$m$$ chỉ số centroid (mỗi chỉ số một byte khi
$$k_s = 256$$). Khoảng cách từ query $$q$$ (không nén) tới vector đã nén $$c$$ tính bằng **ADC** (asymmetric distance
computation):

$$
\tilde{d}(q, c)^2 = \sum_{j=1}^{m} \big\lVert q^{(j)} - C_j[c_j] \big\rVert^2
$$

với bảng $$T_j[s] = \lVert q^{(j)} - C_j[s]\rVert^2$$ tính một lần cho mỗi query ($$m \times k_s$$ giá trị); mỗi vector của
corpus chỉ còn $$m$$ phép tra bảng và cộng.

- `pq_train(x, m, ks=256, n_iter=15, seed=0) -> np.ndarray` shape `(m, ks, d // m)` float32. `ValueError` nếu `d` không
  chia hết cho `m`. Đoạn `j` học bằng Lloyd k-means trên `x[:, j*ds:(j+1)*ds]`, khởi tạo **đúng** như bài Basic 02 với
  `rng = np.random.default_rng(seed + j)`: `rng.choice(n, ks, replace=False)`; đúng `n_iter` vòng, cụm rỗng giữ centroid.
- `pq_encode(x, codebooks) -> np.ndarray` `(n, m)` `uint8`: centroid gần nhất (L2) trong từng đoạn, bằng nhau thì chỉ số nhỏ.
- `pq_decode(codes, codebooks) -> np.ndarray` `(n, d)` float32: ghép các centroid.
- `pq_search(q, codes, codebooks, k) -> list[int]`: top-k theo $$\tilde{d}$$ tăng dần (bằng nhau thì chỉ số nhỏ), tính bằng
  bảng ADC, không giải nén cả corpus.
- `pareto_front(rows) -> list[dict]`: mỗi `row` có khoá `"recall"` (lớn hơn tốt hơn), `"ms"` và `"bytes"` (nhỏ hơn tốt
  hơn). Giữ các dòng **không bị trội**: không có dòng nào khác tốt hơn hoặc bằng ở cả ba và tốt hơn hẳn ở ít nhất một. Giữ
  thứ tự đầu vào.

In [ ]:
def _kmeans(x: np.ndarray, ks: int, n_iter: int, seed: int) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32)
    rng = np.random.default_rng(seed)
    c = x[rng.choice(len(x), ks, replace=False)].copy()
    for _ in range(n_iter):
        d = (x * x).sum(1)[:, None] + (c * c).sum(1)[None, :] - 2 * (x @ c.T)
        lab = np.argmin(d, axis=1)
        for j in range(ks):
            members = x[lab == j]
            if len(members):
                c[j] = members.mean(axis=0)
    return c


def pq_train(x: np.ndarray, m: int, ks: int = 256, n_iter: int = 15, seed: int = 0) -> np.ndarray:
    """Codebook (m, ks, d // m): k-means trên từng đoạn, đoạn j khởi tạo bằng default_rng(seed + j)."""
    # TODO
    raise NotImplementedError


def pq_encode(x: np.ndarray, codebooks: np.ndarray) -> np.ndarray:
    """Mã uint8 (n, m): centroid gần nhất trong từng đoạn."""
    # TODO
    raise NotImplementedError


def pq_decode(codes: np.ndarray, codebooks: np.ndarray) -> np.ndarray:
    """Ghép centroid theo mã: (n, d) float32."""
    # TODO
    raise NotImplementedError


def pq_search(q: np.ndarray, codes: np.ndarray, codebooks: np.ndarray, k: int) -> list[int]:
    """Top-k theo khoảng cách ADC, tăng dần, bằng nhau thì chỉ số nhỏ."""
    # TODO
    raise NotImplementedError


def pareto_front(rows: list[dict]) -> list[dict]:
    """Các dòng không bị trội theo (recall lớn, ms nhỏ, bytes nhỏ), giữ thứ tự."""
    # TODO
    raise NotImplementedError

## 2. Đo (phần của bạn)

Gợi ý khung, chi tiết và tiêu chí ở `README.md` của bài:

1. Nạp ma trận cache, embed 300 câu hỏi ghim, ground truth top-10 bằng kNN chính xác; nạp qrels để tính thêm Recall@10
   theo nhãn.
2. PQ tự viết: train trên một mẫu (ví dụ 20 000 vector), thử `m` = 16, 32, 64; đo recall ANN@10, ms/câu, byte/vector;
   thêm **rescoring**: lấy top-100 theo ADC rồi xếp lại bằng vector gốc.
3. FAISS: `IndexFlatIP`, `IndexHNSWFlat` (vài `efSearch`), `IndexIVFFlat` (vài `nprobe`), `IndexPQ`, `IndexIVFPQ`, và
   `IndexScalarQuantizer` (8 bit) nếu muốn. Đo cùng cách (4 luồng, từng câu một hoặc cả lô, ghi rõ).
4. Bộ nhớ: `faiss.serialize_index(index).nbytes` hoặc công thức, chia cho số vector.
5. `pareto_front`, vẽ recall theo ms và theo byte, rồi điền `REPORT.md`.